## 파이토치를 활용한 회귀 

In [1]:
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, random_split, DataLoader

In [2]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'

torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

In [3]:
df = pd.read_csv('house_prices.csv')
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [4]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [5]:
X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.float32).to(device)  # 회귀문제이므로 float32로 해주기 

In [6]:
dataset = TensorDataset(X, y)

In [7]:
n = len(dataset)
ratio_tn = 0.8
n_tn = int(n*ratio_tn)
n_te = n - n_tn

In [8]:
print("전체 데이터 개수: ", n)
print("트레이닝 데이터 비율: ", ratio_tn)
print("트레이닝 데이터 개수: ", n_tn)
print("테스트 데이터 개수: ", n_te)

전체 데이터 개수:  506
트레이닝 데이터 비율:  0.8
트레이닝 데이터 개수:  404
테스트 데이터 개수:  102


In [9]:
train_data, test_data = random_split(dataset, [n_tn, n_te])

In [10]:
train_data[0]

(tensor([3.4940e-01, 0.0000e+00, 9.9000e+00, 0.0000e+00, 5.4400e-01, 5.9720e+00,
         7.6700e+01, 3.1025e+00, 4.0000e+00, 3.0400e+02, 1.8400e+01, 3.9624e+02,
         9.9700e+00]),
 tensor(20.3000))

In [11]:
batch_size = 16
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

In [22]:
# batch_count = 0
# for batch in train_loader:
#     features, labels = batch
#     print("Features: ", features)
#     print("Labels: ", labels)
#     print()

#     batch_count += 1
#     if batch_count >= 2:
#         break 

In [13]:
# hyperparameters
learning_rate = 0.01
epochs = 1000
n_feature = len(train_data[0][0])
print(n_feature)

13


In [14]:
class Regressor(nn.Module):
    def __init__(self, n_feature):
        super().__init__()
        self.fc1 = nn.Linear(n_feature, 30)
        self.bn1 = torch.nn.BatchNorm1d(30)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(30, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [15]:
# class Regressor(nn.Module):
#     def __init__(self, n_feature):
#         super().__init__()
#         self.fc1 = nn.Linear(n_feature, 30)
#         self.bn1 = torch.nn.BatchNorm1d(30)
#         self.relu = nn.ReLU()
#         self.dropout = nn.Dropout(p=0.3)
#         self.fc2 = nn.Linear(30, 20)
#         self.leakyrelu = nn.LeakyReLU()
#         self.dropout2 = nn.Dropout(p=0)
#         self.fc3 = nn.Linear(20, 1)

#     def forward(self, x):
#         x = self.fc1(x)
#         x = self.bn1(x)
#         x = self.relu(x)
#         x = self.dropout(x)
#         x = self.fc2(x)
#         x = self.leakyrelu(x)
#         x = self.dropout2(x)
#         x = self.fc3(x)
#         return x

In [16]:
model = Regressor(n_feature).to(device)

In [17]:
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [18]:
pre_avg_cost = 1000000

total_batch = len(train_loader)
model.train()
for epoch in range(epochs):
    avg_cost = 0
    for X_tn, y_tn in train_loader:
        X_tn = X_tn.to(device)
        y_tn = y_tn.to(device)

        optimizer.zero_grad()
        y_hat = model.forward(X_tn)
        y_hat = y_hat.squeeze()
        cost = criterion(y_hat, y_tn)
        cost.backward()
        optimizer.step()

        avg_cost += cost/total_batch
        

    if (epoch+1) % 100 == 0:
        print('Epoch [%d/%d], Cost: %.4f' % (epoch+1, epochs, avg_cost))
        
        pre_avg_cost = avg_cost
        

C:\Users\User\AppData\Local\Temp\ipykernel_16220\1271922968.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  print('Epoch [%d/%d], Cost: %.4f' % (epoch+1, epochs, avg_cost))


Epoch [100/1000], Cost: 56.5683
Epoch [200/1000], Cost: 54.0892
Epoch [300/1000], Cost: 40.6008
Epoch [400/1000], Cost: 44.3243
Epoch [500/1000], Cost: 43.3347
Epoch [600/1000], Cost: 46.8802
Epoch [700/1000], Cost: 33.8908
Epoch [800/1000], Cost: 35.0553
Epoch [900/1000], Cost: 41.5686
Epoch [1000/1000], Cost: 45.5303


In [19]:
X_te = []
y_te = []

for row in test_data:
    feature, target = row
    X_te.append(feature)
    y_te.append(target)

In [20]:
X_te = torch.stack(X_te)
y_te = torch.stack(y_te)

In [21]:
model.eval()
with torch.no_grad():
    prediction = model.forward(X_te)
    prediction = prediction.squeeze()
    mse = criterion(prediction, y_te)
    print('MSE:', mse.item())

MSE: 16.781539916992188
